# Assignment 3 — Task 2: Lie Factor audit
 
Compute Tufte's Lie Factor for every graphic using `vizlib.lie_factor()`. The data contains the ink dimension needed for area encodings.

In [4]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from vizlib import lie_factor, lie_verdict

DATA = Path("data")
lf = pd.read_csv(DATA / "case07_lie_factor.csv")
rows = []
for row in lf.itertuples(index=False):
    value = lie_factor(row.value_start, row.value_end, row.ink_start, row.ink_end,
                       ink_dimension=row.ink_dimension)
    factor, data_effect, graphic_effect = value
    rows.append({"chart": row.chart_id, "data_effect": data_effect,
                 "graphic_effect": graphic_effect, "lie_factor": factor,
                 "verdict": lie_verdict(factor),
                 "ink_dimension": row.ink_dimension})
audit = pd.DataFrame(rows).set_index("chart")
display(audit.round(3))
print("Exactly one honest graphic:", audit.index[audit["verdict"] == "honest"].tolist())

,data_effect,graphic_effect,lie_factor,verdict,ink_dimension
chart,,,,,
A,0.067,5.000,75.00,exaggerates 75.00x,1
B,0.500,1.250,2.50,exaggerates 2.50x,2
C,0.200,0.200,1.00,honest,1
D,0.500,0.125,0.25,understates (0.25),1
E,3.000,15.000,5.00,exaggerates 5.00x,2


Exactly one honest graphic: ['C']


### Interpretation

The honest graphic is the row classified as `honest` (0.95–1.05). Every other row distorts the encoded change: a factor above 1 exaggerates it and a factor below 1 understates it. For icon or area encodings, the area is proportional to the square of the linear ink ratio, so using dimension 1 would understate the distortion.

In [5]:
display(lf[["chart_id", "subject", "ink_unit", "ink_dimension",
                        "how_the_graphic_encodes_it"]])
area_cases = lf.loc[lf["ink_dimension"].eq(2), "chart_id"].tolist()
print("Cases requiring ink_dimension=2:", area_cases)
below_one = audit.loc[audit["lie_factor"] < 1]
display(below_one)
print("An understated graphic is still a distortion: it can make a harmful increase look smaller and benefit the publisher's preferred narrative.")

,chart_id,subject,ink_unit,ink_dimension,how_the_graphic_encodes_it
0,A,Corporate tax rate,mm,1,"Bars drawn from a 29.6% baseline, so only the ..."
1,B,Federal health budget (Rs bn),x scale,2,A coin icon scaled 1.5x in BOTH height and width
2,C,Export earnings ($ bn),cm,1,Bars from zero: 5.0 cm then 6.0 cm
3,D,Air-pollution index,cm,1,Bars drawn 8.0 cm then 9.0 cm tall
4,E,City population (millions),x radius,2,A bubble whose RADIUS was set proportional to ...


Cases requiring ink_dimension=2: ['B', 'E']


,data_effect,graphic_effect,lie_factor,verdict,ink_dimension
chart,,,,,
C,0.2,0.200,1.00,honest,1
D,0.5,0.125,0.25,understates (0.25),1


An understated graphic is still a distortion: it can make a harmful increase look smaller and benefit the publisher's preferred narrative.


### Chart F: the dual-axis exception

The Part A dual-axis chart cannot receive a meaningful Lie Factor. The formula assumes that the visual effect and data effect use one shared encoding and one baseline. Two y-axes provide two incompatible scales, so there is no single ink ratio to compare. The raw data itself should be plotted on one common PKR-millions axis.

In [6]:
series = pd.read_csv(DATA / "case07_dual_axis_series.csv")
series["margin_pkr_m"] = series["revenue_pkr_m"] - series["cost_pkr_m"]
display(series)
print("Revenue exceeds cost in every month:", bool((series["margin_pkr_m"] > 0).all()))

,month,revenue_pkr_m,cost_pkr_m,margin_pkr_m
0,Jan,52,44,8
1,Feb,58,46,12
2,Mar,61,47,14
3,Apr,66,49,17
4,May,71,51,20
5,Jun,78,53,25


Revenue exceeds cost in every month: True


### Submission checklist

- The audit calls `vizlib.lie_factor()` for A–E.
- The verdict is computed, not guessed from the prose description.
- Area cases use the supplied `ink_dimension`.
- The dual-axis chart is explained rather than assigned a false number.